# Phase 1 — NCAA MBB Data Pull & Validation

**Goal:** Pull one season (2024) of play-by-play, team box score, and schedule data from the
[sportsdataverse](https://github.com/sportsdataverse) parquet CDN (ESPN-derived data).
Validate columns and a few rows before scaling to 2015–2024.

---

## 0 — Setup & Config

In [34]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import os, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 200)

SEASON = 2024

# sportsdataverse hosts pre-built parquet files on GitHub releases
BASE_URL = "https://github.com/sportsdataverse/sportsdataverse-data/releases/download"

PBP_URL       = f"{BASE_URL}/espn_mens_college_basketball_pbp/play_by_play_{SEASON}.parquet"
TEAM_BOX_URL  = f"{BASE_URL}/espn_mens_college_basketball_team_boxscores/team_box_{SEASON}.parquet"
# Corrected filename for MBB schedule
SCHEDULE_URL  = f"{BASE_URL}/espn_mens_college_basketball_schedules/mbb_schedule_{SEASON}.parquet"

DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Season: {SEASON}")
print(f"Data dir: {DATA_DIR}")
print(f"pandas {pd.__version__}, pyarrow {pa.__version__}")

Season: 2024
Data dir: c:\Users\patgd\Downloads\Github\ncaam_ou_final_mins\data
pandas 2.3.3, pyarrow 23.0.0


## Helper — Download parquet if not cached

In [35]:
import urllib.request

def download_parquet(url, filename):
    """Download a parquet file from URL, cache locally, return as DataFrame."""
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"Downloading from {url} ...")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"  -> saved to {filepath}")
        except Exception as e:
            print(f"  FAILED to download {filename}: {e}")
            return pd.DataFrame()  # Return empty for safety
    else:
        print(f"Using cached {filename}")
    
    # Bypass the pandas.read_parquet wrapper which often crashes due to extension type conflicts.
    # Loading via pyarrow.parquet directly is much more stable in mixed-version environments.
    try:
        table = pq.read_table(filepath)
        df = table.to_pandas()
        print(f"  Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
        return df
    except Exception as e:
        print(f"  Failed to read {filename} with pyarrow: {e}")
        print("  Attempting pandas fallback...")
        return pd.read_parquet(filepath)

---
## 1 — Play-by-Play Data

In [36]:
pbp = download_parquet(PBP_URL, f"pbp_{SEASON}.parquet")

Using cached pbp_2024.parquet
  Shape: 2,004,997 rows x 56 cols


In [37]:
if not pbp.empty:
    print("=== PBP Columns ===")
    for i, col in enumerate(pbp.columns):
        print(f"  {i:3d}  {col}")
else:
    print("PBP data is empty - check download step")

=== PBP Columns ===
    0  game_play_number
    1  id
    2  sequence_number
    3  type_id
    4  type_text
    5  text
    6  away_score
    7  home_score
    8  period_number
    9  period_display_value
   10  clock_display_value
   11  scoring_play
   12  score_value
   13  team_id
   14  athlete_id_1
   15  wallclock
   16  shooting_play
   17  game_id
   18  season
   19  season_type
   20  home_team_id
   21  home_team_name
   22  home_team_mascot
   23  home_team_abbrev
   24  home_team_name_alt
   25  away_team_id
   26  away_team_name
   27  away_team_mascot
   28  away_team_abbrev
   29  away_team_name_alt
   30  game_spread
   31  home_favorite
   32  game_spread_available
   33  home_team_spread
   34  half
   35  time
   36  clock_minutes
   37  clock_seconds
   38  home_timeout_called
   39  away_timeout_called
   40  lead_period
   41  lead_half
   42  start_period_seconds_remaining
   43  start_game_seconds_remaining
   44  end_period_seconds_remaining
   45  end_game_

In [38]:
if not pbp.empty:
    # Critical columns for the endgame analysis (from task.txt)
    REQUIRED_PBP_COLS = [
        'game_id',
        'period_number',
        'clock_display_value',
        'home_score',
        'away_score',
        'scoring_play',
        'score_value',
        'type_text',
        'text',
    ]

    print("=== Critical Column Check ===")
    for col in REQUIRED_PBP_COLS:
        present = col in pbp.columns
        status = "OK" if present else "MISSING"
        print(f"  {status:>7}  {col}")

    missing = [c for c in REQUIRED_PBP_COLS if c not in pbp.columns]
    if missing:
        print(f"\n  Missing columns: {missing}")
        print("Available columns that look similar:")
        for m in missing:
            candidates = [c for c in pbp.columns if m.split('_')[0] in c.lower()]
            print(f"  {m} -> {candidates}")
    else:
        print("\n  All critical PBP columns present!")

=== Critical Column Check ===
       OK  game_id
       OK  period_number
       OK  clock_display_value
       OK  home_score
       OK  away_score
       OK  scoring_play
       OK  score_value
       OK  type_text
       OK  text

  All critical PBP columns present!


In [39]:
if not pbp.empty:
    # Show sample rows - pick one game and display event flow
    sample_game_id = pbp['game_id'].iloc[0]
    sample_game = pbp[pbp['game_id'] == sample_game_id]

    # Select the columns that exist from our required list, plus a few extras
    show_cols = [c for c in REQUIRED_PBP_COLS if c in pbp.columns]
    # Also try to include team-related columns
    team_cols = [c for c in pbp.columns if 'team' in c.lower() and 'id' in c.lower()]
    show_cols = show_cols + [c for c in team_cols if c not in show_cols]

    print(f"\n=== Sample Game: {sample_game_id} ({len(sample_game)} events) ===")
    print(f"Showing first 10 rows with key columns:")
    display(sample_game[show_cols].head(10))


=== Sample Game: 401638645 (294 events) ===
Showing first 10 rows with key columns:


,game_id,period_number,clock_display_value,home_score,away_score,scoring_play,score_value,type_text,text,team_id,home_team_id,away_team_id
0,401638645,1,20:00,0,0,False,0,Jumpball,Jump Ball won by Purdue,2509.0,41,2509
1,401638645,1,19:33,0,0,False,2,JumpShot,Zach Edey missed Jumper.,2509.0,41,2509
2,401638645,1,19:33,0,0,False,0,Defensive Rebound,Stephon Castle Defensive Rebound.,41.0,41,2509
3,401638645,1,19:27,0,0,False,0,Jumpball,Jump Ball won by UConn,41.0,41,2509
4,401638645,1,19:15,0,0,False,3,JumpShot,Alex Karaban missed Three Point Jumper.,41.0,41,2509
5,401638645,1,19:15,0,0,False,0,Defensive Rebound,Braden Smith Defensive Rebound.,2509.0,41,2509
6,401638645,1,19:00,0,2,True,2,JumpShot,Trey Kaufman-Renn made Jumper.,2509.0,41,2509
7,401638645,1,18:40,3,2,True,3,JumpShot,Cam Spencer made Three Point Jumper....,41.0,41,2509
8,401638645,1,18:12,3,4,True,2,LayUpShot,Lance Jones made Layup.,2509.0,41,2509
9,401638645,1,18:12,3,4,False,0,PersonalFoul,Foul on Tristen Newton.,41.0,41,2509


In [40]:
if not pbp.empty:
    # Check the last 10 events of the same game (endgame behavior)
    print(f"=== Last 10 events of game {sample_game_id} (2nd half) ===")
    half2 = sample_game[sample_game['period_number'] == 2] if 'period_number' in sample_game.columns else sample_game
    display(half2[show_cols].tail(10))

=== Last 10 events of game 401638645 (2nd half) ===


,game_id,period_number,clock_display_value,home_score,away_score,scoring_play,score_value,type_text,text,team_id,home_team_id,away_team_id
284,401638645,2,1:47,71,58,True,2,DunkShot,Zach Edey made Dunk.,2509.0,41,2509
285,401638645,2,1:31,71,58,False,0,PersonalFoul,Foul on Braden Smith.,2509.0,41,2509
286,401638645,2,1:31,72,58,True,1,MadeFreeThrow,Stephon Castle made Free Throw.,41.0,41,2509
287,401638645,2,1:31,73,58,True,1,MadeFreeThrow,Stephon Castle made Free Throw.,41.0,41,2509
288,401638645,2,1:22,73,60,True,2,DunkShot,Zach Edey made Dunk. Assisted by Bra...,2509.0,41,2509
289,401638645,2,0:45,75,60,True,2,LayUpShot,Donovan Clingan made Layup. Assisted...,41.0,41,2509
290,401638645,2,0:37,75,60,False,3,JumpShot,Braden Smith missed Three Point Jumper.,2509.0,41,2509
291,401638645,2,0:37,75,60,False,0,Defensive Rebound,UConn Defensive Rebound.,41.0,41,2509
292,401638645,2,0:06,75,60,False,0,Lost Ball Turnover,UConn Turnover.,41.0,41,2509
293,401638645,2,0:00,75,60,False,0,End Game,End of Game,NaN,41,2509


---
## 2 — Team Box Score Data

In [41]:
team_box = download_parquet(TEAM_BOX_URL, f"team_box_{SEASON}.parquet")

Using cached team_box_2024.parquet
  Shape: 12,480 rows x 57 cols


In [42]:
if not team_box.empty:
    print("=== Team Box Score Columns ===")
    for i, col in enumerate(team_box.columns):
        print(f"  {i:3d}  {col}")

=== Team Box Score Columns ===
    0  game_id
    1  season
    2  season_type
    3  game_date
    4  game_date_time
    5  team_id
    6  team_uid
    7  team_slug
    8  team_location
    9  team_name
   10  team_abbreviation
   11  team_display_name
   12  team_short_display_name
   13  team_color
   14  team_alternate_color
   15  team_logo
   16  team_home_away
   17  team_score
   18  team_winner
   19  assists
   20  blocks
   21  defensive_rebounds
   22  fast_break_points
   23  field_goal_pct
   24  field_goals_made
   25  field_goals_attempted
   26  flagrant_fouls
   27  fouls
   28  free_throw_pct
   29  free_throws_made
   30  free_throws_attempted
   31  largest_lead
   32  offensive_rebounds
   33  points_in_paint
   34  steals
   35  team_turnovers
   36  technical_fouls
   37  three_point_field_goal_pct
   38  three_point_field_goals_made
   39  three_point_field_goals_attempted
   40  total_rebounds
   41  total_technical_fouls
   42  total_turnovers
   43  turnover

In [43]:
if not team_box.empty:
    # Columns needed for defensive efficiency:
    # FGA, offensive rebounds (OR), turnovers (TO), FTA -> possessions estimate
    # Points allowed -> defensive efficiency
    DEF_EFFICIENCY_COLS = [
        'game_id', 'team_id', 'team_display_name',
        'field_goals_attempted', 'offensive_rebounds',
        'turnovers', 'free_throws_attempted',
        'team_score', 'opponent_team_score',
    ]

    # Check which exist
    print("=== Defensive Efficiency Column Check ===")
    for col in DEF_EFFICIENCY_COLS:
        present = col in team_box.columns
        status = "OK" if present else "MISSING"
        print(f"  {status:>7}  {col}")

    missing_box = [c for c in DEF_EFFICIENCY_COLS if c not in team_box.columns]
    if missing_box:
        print(f"\n  Missing: {missing_box}")
        print("\nSearching for similar columns...")
        for m in missing_box:
            key = m.split('_')[0]
            candidates = [c for c in team_box.columns if key in c.lower()]
            print(f"  {m} -> {candidates}")

=== Defensive Efficiency Column Check ===
       OK  game_id
       OK  team_id
       OK  team_display_name
       OK  field_goals_attempted
       OK  offensive_rebounds
       OK  turnovers
       OK  free_throws_attempted
       OK  team_score
       OK  opponent_team_score


In [44]:
if not team_box.empty:
    # Sample rows
    print(f"=== Sample Team Box Rows (first 6) ===")
    available_cols = [c for c in DEF_EFFICIENCY_COLS if c in team_box.columns]
    display(team_box[available_cols].head(6))

=== Sample Team Box Rows (first 6) ===


,game_id,team_id,team_display_name,field_goals_attempted,offensive_rebounds,turnovers,free_throws_attempted,team_score,opponent_team_score
0,401638645,2509,Purdue Boilermakers,54,9,9,15,60,75
1,401638645,41,UConn Huskies,62,14,8,11,75,60
2,401638644,333,Alabama Crimson Tide,58,8,8,11,72,86
3,401638644,41,UConn Huskies,62,12,4,18,86,72
4,401638643,152,NC State Wolfpack,57,6,11,4,50,63
5,401638643,2509,Purdue Boilermakers,55,11,16,10,63,50


---
## 3 — Schedule Data

In [45]:
schedule = download_parquet(SCHEDULE_URL, f"schedule_{SEASON}.parquet")

  -> saved to c:\Users\patgd\Downloads\Github\ncaam_ou_final_mins\data\schedule_2024.parquet
  Shape: 6,249 rows x 84 cols


In [46]:
if not schedule.empty:
    print("=== Schedule Columns ===")
    for i, col in enumerate(schedule.columns):
        print(f"  {i:3d}  {col}")

=== Schedule Columns ===
    0  id
    1  uid
    2  date
    3  attendance
    4  time_valid
    5  neutral_site
    6  conference_competition
    7  play_by_play_available
    8  recent
    9  start_date
   10  notes_type
   11  notes_headline
   12  broadcast_market
   13  broadcast_name
   14  type_id
   15  type_abbreviation
   16  venue_id
   17  venue_full_name
   18  venue_address_city
   19  venue_address_state
   20  venue_indoor
   21  status_clock
   22  status_display_clock
   23  status_period
   24  status_type_id
   25  status_type_name
   26  status_type_state
   27  status_type_completed
   28  status_type_description
   29  status_type_detail
   30  status_type_short_detail
   31  format_regulation_periods
   32  home_id
   33  home_uid
   34  home_location
   35  home_name
   36  home_abbreviation
   37  home_display_name
   38  home_short_display_name
   39  home_color
   40  home_alternate_color
   41  home_is_active
   42  home_venue_id
   43  home_logo
   44  ho

In [47]:
if not schedule.empty:
    # Check for home/away/neutral, conference info
    SCHEDULE_COLS = [
        'game_id',
        'home_id', 'away_id',
        'home_display_name', 'away_display_name',
        'conference_id', 'groups_name',
        'season',
        'neutral_site',
        'season_type',
    ]

    print("=== Schedule Column Check ===")
    for col in SCHEDULE_COLS:
        present = col in schedule.columns
        status = "OK" if present else "MISSING"
        print(f"  {status:>7}  {col}")

    missing_sched = [c for c in SCHEDULE_COLS if c not in schedule.columns]
    if missing_sched:
        print(f"\n  Missing: {missing_sched}")
        print("\nSearching for similar columns...")
        for m in missing_sched:
            key = m.split('_')[0]
            candidates = [c for c in schedule.columns if key in c.lower()]
            print(f"  {m} -> {candidates}")

=== Schedule Column Check ===
       OK  game_id
       OK  home_id
       OK  away_id
       OK  home_display_name
       OK  away_display_name
  MISSING  conference_id
       OK  groups_name
       OK  season
       OK  neutral_site
       OK  season_type

  Missing: ['conference_id']

Searching for similar columns...
  conference_id -> ['conference_competition', 'home_conference_id', 'away_conference_id', 'groups_is_conference']


In [48]:
if not schedule.empty:
    # Sample schedule rows
    available_sched = [c for c in SCHEDULE_COLS if c in schedule.columns]
    print(f"=== Sample Schedule Rows ===")
    display(schedule[available_sched].head(6))

=== Sample Schedule Rows ===


,game_id,home_id,away_id,home_display_name,away_display_name,groups_name,season,neutral_site,season_type
0,401638645,41,2509,UConn Huskies,Purdue Boilermakers,None,2024,True,3
1,401638644,41,333,UConn Huskies,Alabama Crimson Tide,None,2024,True,3
2,401638643,2509,152,Purdue Boilermakers,NC State Wolfpack,None,2024,True,3
3,401641124,2550,282,Seton Hall Pirates,Indiana State Sycamores,None,2024,True,3
4,401641122,2550,61,Seton Hall Pirates,Georgia Bulldogs,None,2024,True,3
5,401641123,282,254,Indiana State Sycamores,Utah Utes,None,2024,True,3


---
## 4 — Null & Coverage Report

In [49]:
def null_report(df, name, key_cols):
    """Print null counts and percentages for key columns."""
    if df.empty:
        print(f"\n{name} is empty -- skipping null report")
        return
    existing = [c for c in key_cols if c in df.columns]
    total = len(df)
    print(f"\n{'='*50}")
    print(f"{name}  --  {total:,} rows")
    print(f"{'='*50}")
    print(f"{'Column':<35} {'Nulls':>8} {'%':>8}")
    print(f"{'-'*35} {'-'*8} {'-'*8}")
    for col in existing:
        nulls = df[col].isna().sum()
        pct = 100 * nulls / total if total > 0 else 0
        flag = ' WARNING' if pct > 5 else ''
        print(f"{col:<35} {nulls:>8,} {pct:>7.1f}%{flag}")

null_report(pbp, "Play-by-Play", REQUIRED_PBP_COLS if not pbp.empty else [])
null_report(team_box, "Team Box Scores", DEF_EFFICIENCY_COLS if not team_box.empty else [])
null_report(schedule, "Schedule", SCHEDULE_COLS if not schedule.empty else [])


Play-by-Play  --  2,004,997 rows
Column                                 Nulls        %
----------------------------------- -------- --------
game_id                                    0     0.0%
period_number                              0     0.0%
clock_display_value                        0     0.0%
home_score                                 0     0.0%
away_score                                 0     0.0%
scoring_play                               0     0.0%
score_value                                0     0.0%
type_text                                  0     0.0%
text                                   1,737     0.1%

Team Box Scores  --  12,480 rows
Column                                 Nulls        %
----------------------------------- -------- --------
game_id                                    0     0.0%
team_id                                    0     0.0%
team_display_name                          0     0.0%
field_goals_attempted                      0     0.0%
offensive_rebo

In [50]:
# Coverage check: how many unique games?
pbp_games = pbp['game_id'].nunique() if not pbp.empty else 0
box_games = team_box['game_id'].nunique() if not team_box.empty and 'game_id' in team_box.columns else 0
sched_games = schedule['game_id'].nunique() if not schedule.empty and 'game_id' in schedule.columns else 0

print(f"\n=== Game Coverage ({SEASON}) ===")
print(f"  PBP unique games:       {pbp_games:,}")
print(f"  Box score unique games: {box_games:,}")
print(f"  Schedule unique games:  {sched_games:,}")

if pbp_games > 3000:
    print("  PBP game count looks reasonable")
elif not pbp.empty:
    print("  WARNING: PBP game count seems low -- investigate")


=== Game Coverage (2024) ===
  PBP unique games:       6,151
  Box score unique games: 6,240
  Schedule unique games:  6,249
  PBP game count looks reasonable


---
## 5 — Quick Sanity: Clock Monotonicity & Score Progression

In [51]:
# Check if clock_display_value can be parsed to seconds
def clock_to_seconds(clock_str):
    """Convert 'MM:SS' clock display to seconds remaining."""
    try:
        if pd.isna(clock_str):
            return None
        parts = str(clock_str).split(':')
        if len(parts) == 2:
            return int(parts[0]) * 60 + int(parts[1])
        return None
    except:
        return None

# Test on sample game
if not pbp.empty and 'clock_display_value' in pbp.columns:
    sample = pbp[pbp['game_id'] == sample_game_id].copy()
    sample['secs_remaining'] = sample['clock_display_value'].apply(clock_to_seconds)
    
    parseable = sample['secs_remaining'].notna().sum()
    total_events = len(sample)
    print(f"Clock parsing success: {parseable}/{total_events} events ({100*parseable/total_events:.1f}%)")
    
    # Show the last few events of 2nd half with parsed clock
    half2_cols = ['period_number', 'clock_display_value', 'home_score', 'away_score', 'type_text', 'text']
    half2_cols = [c for c in half2_cols if c in sample.columns]
    h2 = sample[sample['period_number'] == 2] if 'period_number' in sample.columns else sample
    print(f"\nLast 10 events of 2nd half (game {sample_game_id}):")
    display(h2[half2_cols + ['secs_remaining']].tail(10))
elif not pbp.empty:
    print("WARNING: clock_display_value not found -- check column listing above")

Clock parsing success: 294/294 events (100.0%)

Last 10 events of 2nd half (game 401638645):


,period_number,clock_display_value,home_score,away_score,type_text,text,secs_remaining
284,2,1:47,71,58,DunkShot,Zach Edey made Dunk.,107
285,2,1:31,71,58,PersonalFoul,Foul on Braden Smith.,91
286,2,1:31,72,58,MadeFreeThrow,Stephon Castle made Free Throw.,91
287,2,1:31,73,58,MadeFreeThrow,Stephon Castle made Free Throw.,91
288,2,1:22,73,60,DunkShot,Zach Edey made Dunk. Assisted by Bra...,82
289,2,0:45,75,60,LayUpShot,Donovan Clingan made Layup. Assisted...,45
290,2,0:37,75,60,JumpShot,Braden Smith missed Three Point Jumper.,37
291,2,0:37,75,60,Defensive Rebound,UConn Defensive Rebound.,37
292,2,0:06,75,60,Lost Ball Turnover,UConn Turnover.,6
293,2,0:00,75,60,End Game,End of Game,0


In [52]:
# Check score monotonicity -- scores should only increase
if not pbp.empty and 'home_score' in pbp.columns and 'away_score' in pbp.columns:
    sample = pbp[pbp['game_id'] == sample_game_id].copy()
    sample['home_score'] = pd.to_numeric(sample['home_score'], errors='coerce')
    sample['away_score'] = pd.to_numeric(sample['away_score'], errors='coerce')
    
    home_decreases = (sample['home_score'].diff() < 0).sum()
    away_decreases = (sample['away_score'].diff() < 0).sum()
    
    # Period breaks may cause score 'resets' in display -- check within period
    print(f"Score decrease events (home): {home_decreases}")
    print(f"Score decrease events (away): {away_decreases}")
    if home_decreases <= 1 and away_decreases <= 1:
        print("Score progression looks monotonic (<=1 decrease, likely period boundary)")
    else:
        print("WARNING: Multiple score decreases detected -- investigate data ordering")
elif not pbp.empty:
    print("WARNING: home_score / away_score columns not found")

Score decrease events (home): 0
Score decrease events (away): 0
Score progression looks monotonic (<=1 decrease, likely period boundary)


---
## 6 — Summary & Next Steps

**If all checks pass above**, the data supports the endgame scoring analysis in `task.txt`.

**Next steps (Phase 2):**
1. Scale to 2015-2024: `for season in range(2015, 2025): download_parquet(...)`
2. Parse clock -> seconds remaining in regulation
3. Build margin buckets and exposure table
4. Compute team-season defensive efficiency from box scores
5. Tag foul/FT events for endgame regime detection

In [53]:
pbp_rows = pbp.shape[0] if not pbp.empty else 0
pbp_cols = pbp.shape[1] if not pbp.empty else 0
box_rows = team_box.shape[0] if not team_box.empty else 0
box_cols = team_box.shape[1] if not team_box.empty else 0
sched_rows = schedule.shape[0] if not schedule.empty else 0
sched_cols = schedule.shape[1] if not schedule.empty else 0

print("Phase 1 data pull complete.")
print(f"  PBP:      {pbp_rows:>10,} rows, {pbp_cols:>3} cols, {pbp_games:,} games")
print(f"  Box:      {box_rows:>10,} rows, {box_cols:>3} cols, {box_games:,} games")
print(f"  Schedule: {sched_rows:>10,} rows, {sched_cols:>3} cols, {sched_games:,} games")

Phase 1 data pull complete.
  PBP:       2,004,997 rows,  56 cols, 6,151 games
  Box:          12,480 rows,  57 cols, 6,240 games
  Schedule:      6,249 rows,  84 cols, 6,249 games
